# Comma Attender

Fraction of total attention where comma (,) is the source (attended to)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
    show_type_tokens, show_type_filtered_tables,
    get_type_positions, TYPE_ID_TO_POSITION_KEY,
)

/home/burny/projects/ai/mechinterp/attention-head-zoo-2-layer-attention-only-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Matched Tokens

In [3]:
show_type_tokens(str_tokens, "comma_attention")

**Matched tokens (3):** `,` (5), `,` (33), `,` (48)

## Comma Attender — All 24 Heads

In [4]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("comma_attention")
is_measurable = ("comma_attention", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("comma_attention", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | Comma Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | Comma Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")

| Head | Comma Attender % | Entropy % |
|------|-------|-------|
| L0H5 | 7.37% | 53.32% |
| L1H1 | 7.20% | 72.18% |
| L0H9 | 5.24% | 47.79% |
| L0H7 | 4.83% | 26.74% |
| L0H0 | 4.47% | 86.46% |
| L1H0 | 4.45% | 81.29% |
| L1H11 | 3.54% | 78.88% |
| L0H11 | 3.41% | 64.07% |
| L0H1 | 3.30% | 81.72% |
| L0H4 | 3.29% | 72.07% |
| L1H2 | 3.28% | 57.12% |
| L1H9 | 3.16% | 94.13% |
| L1H3 | 3.05% | 94.52% |
| L1H5 | 2.74% | 86.47% |
| L1H7 | 2.45% | 75.03% |
| L0H8 | 2.28% | 79.84% |
| L0H10 | 2.20% | 81.09% |
| L0H2 | 2.09% | 81.11% |
| L0H6 | 2.01% | 96.14% |
| L1H8 | 1.98% | 78.84% |
| L1H6 | 1.90% | 61.80% |
| L0H3 | 1.47% | 92.93% |
| L1H4 | 0.88% | 71.28% |
| L1H10 | 0.53% | 76.69% |

## Top Heads

In [5]:
pos_key = TYPE_ID_TO_POSITION_KEY.get("comma_attention")
_type_positions = get_type_positions(str_tokens).get(pos_key, []) if pos_key else []
sorted_heads = sorted(
    [((l, h), tm[("comma_attention", l, h)]) for l in range(2) for h in range(12) if ("comma_attention", l, h) in tm],
    key=lambda x: x[1], reverse=True,
)[:3]
for (l, h), pct in sorted_heads:
    ent_str = ""
    if ent_key and (ent_key, l, h) in tm:
        ent_str = f" | ent {tm[(ent_key, l, h)]:.2f}%"
    level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
    display(Markdown(f"---\n### L{l}H{h} — {pct:.2f}% ({level}){ent_str}"))
    show_head_pattern(str_tokens, cache, layer=l, head=h)
    attention = get_attention_pattern(cache, layer=l, head=h)
    if _type_positions:
        show_type_filtered_tables(str_tokens, attention, _type_positions, "Comma Attender", top_k=15)
    show_attention_tables(str_tokens, attention, top_k=25)

---
### L0H5 — 7.37% (almost none) | ent 53.32%

### Highest attention to Comma Attender tokens (dest ← Comma Attender src)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` created` | `,` | 18 | 5 | 0.9609 |
| 2 | ` likely` | `,` | 13 | 5 | 0.7738 |
| 3 | ` known` | `,` | 55 | 48 | 0.6299 |
| 4 | ` think` | `,` | 35 | 33 | 0.4292 |
| 5 | ` scaled` | `,` | 28 | 5 | 0.3297 |
| 6 | ` solid` | `,` | 52 | 48 | 0.2582 |
| 7 | ` deceptive` | `,` | 44 | 33 | 0.2203 |
| 8 | ` known` | `,` | 55 | 33 | 0.1890 |
| 9 | ` avoid` | `,` | 59 | 48 | 0.1618 |
| 10 | ` up` | `,` | 29 | 5 | 0.1413 |
| 11 | ` avoid` | `,` | 59 | 33 | 0.0707 |
| 12 | ` think` | `,` | 35 | 5 | 0.0636 |
| 13 | ` solid` | `,` | 52 | 33 | 0.0624 |
| 14 | ` deceptive` | `,` | 44 | 5 | 0.0449 |
| 15 | ` known` | `,` | 55 | 5 | 0.0429 |

### Highest attention from Comma Attender tokens (Comma Attender dest ← src)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.5008 |
| 2 | `,` | `We` | 5 | 1 | 0.4383 |
| 3 | `,` | ` we` | 48 | 34 | 0.2347 |
| 4 | `,` | ` or` | 48 | 45 | 0.2067 |
| 5 | `,` | `<\|endoftext\|>` | 33 | 0 | 0.1886 |
| 6 | `,` | ` this` | 33 | 31 | 0.1625 |
| 7 | `,` | `We` | 33 | 1 | 0.1296 |
| 8 | `,` | ` are` | 48 | 43 | 0.1289 |
| 9 | `,` | ` to` | 33 | 30 | 0.1198 |
| 10 | `,` | ` be` | 33 | 17 | 0.0796 |
| 11 | `,` | `<\|endoftext\|>` | 48 | 0 | 0.0750 |
| 12 | `,` | ` by` | 48 | 38 | 0.0711 |
| 13 | `,` | ` not` | 33 | 15 | 0.0668 |
| 14 | `,` | ` this` | 33 | 19 | 0.0594 |
| 15 | `,` | ` more` | 33 | 12 | 0.0524 |

...

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `,` | `,` | 33 | 33 | 0.7264 |
| 2 | `,` | `,` | 48 | 48 | 0.7177 |
| 3 | `,` | `,` | 5 | 5 | 0.4452 |
| 4 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.2718 |
| 5 | `,` | ` that` | 5 | 3 | 0.2424 |
| 6 | `,` | ` or` | 48 | 45 | 0.1767 |
| 7 | `,` | `<\|endoftext\|>` | 33 | 0 | 0.1265 |
| 8 | `,` | ` level` | 33 | 32 | 0.0378 |
| 9 | `,` | `ulative` | 48 | 47 | 0.0373 |
| 10 | `,` | ` this` | 33 | 31 | 0.0288 |
| 11 | `,` | ` If` | 33 | 22 | 0.0287 |
| 12 | `,` | `We` | 5 | 1 | 0.0233 |
| 13 | `,` | ` to` | 33 | 30 | 0.0227 |
| 14 | `,` | ` manip` | 48 | 46 | 0.0195 |
| 15 | `,` | `<\|endoftext\|>` | 48 | 0 | 0.0128 |

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` If` | ` If` | 22 | 22 | 0.8735 |
| 3 | ` up` | ` scaled` | 29 | 28 | 0.8550 |
| 4 | ` manip` | ` manip` | 46 | 46 | 0.8482 |
| 5 | ` we` | ` we` | 34 | 34 | 0.8378 |
| 6 | ` default` | ` default` | 39 | 39 | 0.8225 |
| 7 | ` super` | ` super` | 7 | 7 | 0.8133 |
| 8 | ` would` | ` they` | 37 | 36 | 0.7853 |
| 9 | ` and` | `,` | 49 | 48 | 0.7469 |
| 10 | ` to` | ` up` | 30 | 29 | 0.7334 |
| 11 | ` more` | ` is` | 12 | 11 | 0.7281 |
| 12 | `,` | `,` | 33 | 33 | 0.7264 |
| 13 | ` that` | ` systems` | 42 | 41 | 0.7203 |
| 14 | `,` | `,` | 48 | 48 | 0.7177 |
| 15 | ` plans` | ` plans` | 53 | 53 | 0.7089 |
| 16 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.7061 |
| 17 | ` they` | ` they` | 36 | 36 | 0.6669 |
| 18 | ` to` | ` how` | 58 | 57 | 0.6645 |
| 19 | `.` | `.` | 61 | 61 | 0.6587 |
| 20 | ` avoid` | ` avoid` | 59 | 59 | 0.6587 |
| 21 | ` systems` | ` systems` | 41 | 41 | 0.6511 |
| 22 | ` that` | ` and` | 50 | 49 | 0.6431 |
| 23 | ` century` | ` century` | 20 | 20 | 0.6290 |
| 24 | ` than` | ` likely` | 14 | 13 | 0.6034 |
| 25 | ` level` | ` level` | 32 | 32 | 0.5864 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` plans` | `,` | 53 | 5 | 0.0000 |
| 2 | ` and` | ` likely` | 49 | 13 | 0.0000 |
| 3 | ` plans` | ` significantly` | 53 | 6 | 0.0000 |
| 4 | ` up` | `,` | 29 | 5 | 0.0000 |
| 5 | ` and` | ` powerful` | 49 | 4 | 0.0000 |
| 6 | ` plans` | ` is` | 53 | 11 | 0.0000 |
| 7 | ` or` | ` think` | 45 | 2 | 0.0000 |
| 8 | ` are` | ` significantly` | 54 | 6 | 0.0000 |
| 9 | ` plans` | ` that` | 53 | 3 | 0.0000 |
| 10 | ` solid` | `,` | 52 | 5 | 0.0000 |
| 11 | ` that` | ` than` | 50 | 14 | 0.0000 |
| 12 | `,` | ` powerful` | 48 | 4 | 0.0000 |
| 13 | `.` | ` significantly` | 61 | 6 | 0.0000 |
| 14 | ` plans` | ` super` | 53 | 7 | 0.0000 |
| 15 | ` solid` | ` significantly` | 52 | 6 | 0.0000 |
| 16 | ` manip` | ` that` | 46 | 3 | 0.0000 |
| 17 | ` and` | ` think` | 49 | 2 | 0.0000 |
| 18 | `.` | ` machine` | 61 | 9 | 0.0000 |
| 19 | ` plans` | ` powerful` | 53 | 4 | 0.0000 |
| 20 | ` to` | ` likely` | 58 | 13 | 0.0000 |
| 21 | ` solid` | ` powerful` | 52 | 4 | 0.0000 |
| 22 | ` avoid` | ` than` | 59 | 14 | 0.0000 |
| 23 | ` this` | ` significantly` | 60 | 6 | 0.0000 |
| 24 | ` this` | ` than` | 60 | 14 | 0.0000 |
| 25 | ` to` | ` more` | 58 | 12 | 0.0000 |